In [ ]:
# STUDENT PERFORMANCE PREDICTION ML MODEL
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

import matplotlib.pyplot as plt

# DATASET
csv_file = "student_data.csv"

# Default dataset in case CSV doesn't exist
default_data = {
    "study_hours": [2, 4, 6, 8, 1, 5, 7, 3, 9, 4],
    "attendance": [60, 75, 85, 90, 50, 80, 95, 65, 98, 70],
    "previous_marks": [50, 65, 78, 88, 40, 70, 92, 55, 95, 60],
    "assignments_completed": [4, 6, 8, 9, 2, 7, 10, 5, 10, 6],
    "final_score": [55, 68, 80, 92, 45, 75, 96, 60, 99, 65]
}

if not os.path.exists(csv_file):
    # Create default CSV file
    df = pd.DataFrame(default_data)
    df.to_csv(csv_file, index=False)
    print(f"Created default dataset in '{csv_file}'. Feel free to edit this file with your own dataset!")
else:
    # Read existing CSV file
    df = pd.read_csv(csv_file)
    print(f"Successfully loaded dataset from '{csv_file}' containing {len(df)} records.")

# Show dataset
print("\nDataset:")
print(df)

#  FEATURES AND TARGET

X = df.drop("final_score", axis=1)

y = df["final_score"]

print("\nFeatures:")
print(X.head())

print("\nTarget:")
print(y.head())

# SPLIT DATA

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("\nTraining Data Size:", len(X_train))
print("Testing Data Size:", len(X_test))

# CREATE MODEL
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

#  TRAIN MODEL

model.fit(X_train, y_train)

print("\nModel Training Completed!")

#  MAKE PREDICTIONS

predictions = model.predict(X_test)

print("\nPredictions:")
print(predictions)

#EVALUATE MODEL

mae = mean_absolute_error(y_test, predictions)

r2 = r2_score(y_test, predictions)

print("\nMean Absolute Error:", round(mae, 2))

print("R2 Score:", round(r2, 2))

# TEST CUSTOM STUDENT
print("\n--- Predict Score for a Custom Student ---")

# We use default values as fallback
custom_student = {
    "study_hours": 6,
    "attendance": 85,
    "previous_marks": 75,
    "assignments_completed": 8
}

try:
    print("Enter your own student data below (or press Enter to use default values):")
    
    sh_input = input(f"Study Hours [Default: {custom_student['study_hours']}]: ").strip()
    if sh_input:
        custom_student["study_hours"] = float(sh_input)
        
    att_input = input(f"Attendance (%) [Default: {custom_student['attendance']}]: ").strip()
    if att_input:
        custom_student["attendance"] = float(att_input)
        
    pm_input = input(f"Previous Marks [Default: {custom_student['previous_marks']}]: ").strip()
    if pm_input:
        custom_student["previous_marks"] = float(pm_input)
        
    ac_input = input(f"Assignments Completed [Default: {custom_student['assignments_completed']}]: ").strip()
    if ac_input:
        custom_student["assignments_completed"] = float(ac_input)
except (EOFError, IOError, ValueError) as e:
    print("Using default values for prediction due to non-interactive environment or invalid entry.")

student = pd.DataFrame({
    "study_hours": [custom_student["study_hours"]],
    "attendance": [custom_student["attendance"]],
    "previous_marks": [custom_student["previous_marks"]],
    "assignments_completed": [custom_student["assignments_completed"]]
})

predicted_score = model.predict(student)

print("\nPredicted Student Score:")
print(f"Parameters: {custom_student}")
print(f"Predicted Score: {round(predicted_score[0], 2)}")

# FEATURE IMPORTANCE
importance = model.feature_importances_

features = X.columns

importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": importance
})

print("\nFeature Importance:")
print(importance_df)

# graphs and charts

plt.figure(figsize=(8,5))

plt.bar(features, importance)

plt.xlabel("Features")

plt.ylabel("Importance")

plt.title("Feature Importance in Student Performance Prediction")

plt.xticks(rotation=15)

plt.show()

#  SAVE MODEL

import joblib

joblib.dump(model, "student_performance_model.pkl")

print("\nModel Saved Successfully!")
